# Определение математических заблуждений — обучение модели

Здесь я обучаю модель, которая по объяснению ученика определяет тип ошибки. В конце сохраняю модель и словарь ответов для веб-приложения (Hugging Face).

In [ ]:
# Шаг 1. Загружаем датасет в сессию Colab.
# Сессия обнуляется при перезапуске, поэтому файл заливаем заново.
from google.colab import files
uploaded = files.upload()   # выбери train.csv

In [ ]:
# Шаг 2. Читаем таблицу.
# ВАЖНО: путь без "/" в начале — файл лежит в текущей папке Colab.
import pandas as pd

df = pd.read_csv("train.csv", on_bad_lines="skip")

print("Размер таблицы:", df.shape)
print("Колонки:", df.columns.tolist())
df.head()

In [ ]:
# Шаг 3. Готовим данные.
# Подгони имена колонок под свой файл (проверь по списку выше).
TEXT_QUESTION = "QuestionText"        # текст задачи
TEXT_EXPLAIN  = "StudentExplanation"  # объяснение ученика
TARGET_COL    = "Misconception"       # тип ошибки (наша цель)

# у правильных ответов ошибки нет -> пустые значения = "No_Misconception"
df[TARGET_COL] = df[TARGET_COL].fillna("No_Misconception")

# собираем текст так же, как потом будет в приложении: вопрос + объяснение
df["text"] = (df[TEXT_QUESTION].fillna("") + " " + df[TEXT_EXPLAIN].fillna("")).str.strip()

X = df["text"]
y = df[TARGET_COL]

print("Всего примеров:", len(df))
print("Сколько классов:", y.nunique())
print("\nСамые частые классы:")
print(y.value_counts().head(10))

In [ ]:
# Шаг 4. Делим на train/test.
# Проблема: один и тот же вопрос встречается у многих учеников. Если он попадёт
# и в train, и в test — модель "подсмотрит" ответ. Поэтому делим ПО ВОПРОСАМ.
from sklearn.model_selection import train_test_split, GroupShuffleSplit

if "QuestionId" in df.columns:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups=df["QuestionId"]))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    print("Деление по вопросам (без утечки).")
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print("Обычное деление со стратификацией.")

print("Обучение:", len(X_train), "| Тест:", len(X_test))

In [ ]:
# Шаг 5. BASELINE — "отметка на стене".
# Простейшая модель: всегда предсказывает самый частый класс.
# Если моя модель не лучше неё — значит, она бесполезна.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)

print("BASELINE")
print("  Accuracy:", round(accuracy_score(y_test, base_pred), 3))
print("  macro-F1:", round(f1_score(y_test, base_pred, average="macro"), 3))

In [ ]:
# Шаг 6. Моя модель: TF-IDF + логистическая регрессия, всё в одном Pipeline.
# TF-IDF превращает текст в числа, LogReg предсказывает класс.
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

model = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2))),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("МОЯ МОДЕЛЬ")
print("  Accuracy:", round(accuracy_score(y_test, pred), 3))
print("  macro-F1:", round(f1_score(y_test, pred, average="macro"), 3))
print("\n(эти числа впиши в таблицу метрик в README)")

# подробный отчёт по классам
print("\n", classification_report(y_test, pred, zero_division=0))

In [ ]:
# Шаг 7. Сохраняем модель и русские ответы для веб-приложения, затем скачиваем.
import joblib
from google.colab import files

answers = {'No_Misconception': 'Ошибок не обнаружено. Объяснение ученика выглядит корректным.', 'Incomplete': 'Объяснение неполное — пропущена важная часть рассуждения.', 'Wrong_fraction': 'Возможна ошибка при работе с дробями.', 'Wrong_Fraction': 'Возможна ошибка при работе с дробями.', 'Wrong_term': 'Возможно, математический термин используется неверно.', 'Additive': 'Возможно, применяется сложение там, где нужна другая операция.', 'Subtraction': 'Возможно, неверно применяется вычитание.', 'Division': 'Возможно, неверно применяется деление.', 'Mult': 'Возможно, неверно применяется умножение.', 'Inversion': 'Возможно, неверно понято обратное действие или преобразование.', 'Duplication': 'Возможно, одно и то же значение учитывается или применяется дважды.', 'Positive': 'Возможно, неверно понят знак или положительное значение.', 'Scale': 'Ошибка может быть связана с масштабом — изменением величины.', 'Whole_numbers_larger': 'Возможно, ученик считает, что большее целое число всегда означает большую величину.', 'Not_variable': 'Возможно, неверно понята роль переменной.', 'Wrong_Operation': 'Возможно, выбрана неверная математическая операция.', 'WNB': 'Рассуждение может быть неверным или недостаточным.', 'Irrelevant': 'В объяснении есть информация, не относящаяся к решению задачи.', 'Unknowable': 'Возможно, ученик считает, что ответ нельзя определить из данных условия.', 'Adding_across': 'Ученик складывает числители и знаменатели вместо приведения к общему знаменателю.'}

joblib.dump(model, "math_misconception_model.pkl")
joblib.dump(answers, "answers.pkl")

files.download("math_misconception_model.pkl")
files.download("answers.pkl")
print("Готово — скачай оба файла и залей в Space рядом с app.py")